# Iteration 4 — primary PySpark preprocessing and holdout workflow
Use this notebook in Google Colab to run native Spark logistic regression, random forest and GBT. It compares **all predictors, CART20, RFE20 and RFE10**, with continuous standardization, original ordinal codes, five 70/30 stratified holdouts and three-fold tuning only inside training.

Selection is implemented as a Spark pipeline estimator, refitted within every CV fit. Native Spark tree importances drive CART selection and one-at-a-time RFE. The 20/10 controls count original source fields. The existing five-family Python/Optuna notebook remains `run_python_workflow.ipynb`.

The report includes the author's saved 15-model Spark baseline execution. Scores for the new selected-field Spark experiments are reported only after completing this notebook; no historical Python scores are labelled as native Spark results.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, shutil
missing=[spec for module,spec in {'pyspark':'pyspark>=3.5.3,<4.0','sklearn':'scikit-learn>=1.3'}.items()
         if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
java=shutil.which('java')
if not java or subprocess.run([java,'-version'],capture_output=True).returncode:
    raise RuntimeError('Use a runtime with working Java before starting Spark.')
# Local standalone Iteration 4 folder, or the GitHub repository checkout.
if (Path.cwd()/'spark_iterations.py').is_file() and (Path.cwd()/'data/raw/dataset_gallstone').is_file():
    SPARK_DIR=Path.cwd();DATA_PATH=SPARK_DIR/'data/raw/dataset_gallstone'
else:
    candidates=[Path.cwd(),Path.cwd().parent,Path('/content/INFOSYS722_BDAS')]
    REPO_ROOT=next((p for p in candidates if (p/'PySpark/spark_iterations.py').is_file()),None)
    if REPO_ROOT is None:
        REPO_ROOT=Path('/content/INFOSYS722_BDAS')
        if REPO_ROOT.exists():
            raise RuntimeError('Update the existing checkout to the latest main, then rerun this cell.')
        subprocess.check_call(['git','clone','https://github.com/QuyDuong106/INFOSYS722_BDAS.git',str(REPO_ROOT)])
    SPARK_DIR=REPO_ROOT/'PySpark';DATA_PATH=REPO_ROOT/'iteration4_BDAS/data/raw/dataset_gallstone'
sys.path.insert(0,str(SPARK_DIR))
print('Spark implementation:',SPARK_DIR)
print('Dataset:',DATA_PATH)


## Validate the Spark feature-selection extension
This checks the retained field count, ordinal-code preservation, saved-model roundtrip and CV/holdout separation before the full experiment. It needs the same Spark/Java runtime.

In [ ]:
RUN_INTEGRATION_CHECKS=True
test_dir=SPARK_DIR/'spark_tests' if (SPARK_DIR/'spark_tests').exists() else SPARK_DIR/'tests'
if RUN_INTEGRATION_CHECKS:
    subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(test_dir),
                           '-p','test_spark_feature_selection.py','-v'])


In [ ]:
from spark_workflow import create_spark,load_dataset
from spark_iterations import run
spark=create_spark()
frame,numeric,categorical=load_dataset(spark,DATA_PATH)
frame.groupBy('label').count().show()
print('Spark:',spark.version,'records:',frame.count())


## Execute controlled preparation iterations
RFE repeatedly fits native Spark trees and can take substantial time. Default execution covers all four variants, three model families and five seeds. Use a subset of `VARIANTS` to run separate batches; identical settings resume completed model checkpoints. Every variant receives the same test membership and training folds. No test record enters selection or tuning.

In [ ]:
VARIANTS=['all_standard','cart20_standard','rfe20_standard','rfe10_standard']
metrics,summary,OUTPUT_DIR=run(spark,frame,numeric,categorical,data_path=DATA_PATH,variants=VARIANTS,tune=True)
print('Completed output directory:',OUTPUT_DIR)
metrics.orderBy('variant','seed','model').show(100,truncate=False)
summary.orderBy('variant','roc_auc_mean',ascending=[True,False]).show(30,truncate=False)


## Save and report the completed run
The output folder contains configuration and code/data signatures, split membership, selected original fields, CV search settings/scores, persisted Spark pipelines, predictions and aggregate JSON/CSV evidence. Download it before ending the runtime, or set an explicit compatible `output_dir` in persistent storage. Use `scripts/build_extension_report.py --spark-results <completed_directory>` from Iteration 4 to add measured native Spark feature-count comparisons to the report. Whole-cohort exploratory plots remain in `PySpark/spark_visualization_colab.ipynb`.

In [ ]:
print('Results:',OUTPUT_DIR)
# Optional download of the aggregate metrics only:
# from google.colab import files
# files.download(str(OUTPUT_DIR/'metrics.json'))
